In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install kaggle

In [ ]:
!curl -L -o intersection-flow-5k.zip https://www.kaggle.com/api/v1/datasets/download/starsw/intersection-flow-5k

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 5889M  100 5889M    0     0   162M      0  0:00:36  0:00:36 --:--:--  197M


In [ ]:
!unzip /content/intersection-flow-5k.zip

Streaming output truncated to the last 5000 lines.
  inflating: Intersection-Flow-5K/labels/train/1662004713.274337053.txt  
  inflating: Intersection-Flow-5K/labels/train/1662004758.668909788.txt  
  inflating: Intersection-Flow-5K/labels/train/1662004761.170110464.txt  
  inflating: Intersection-Flow-5K/labels/train/1662004762.470435619.txt  
  inflating: Intersection-Flow-5K/labels/train/1662004765.370779037.txt  
  inflating: Intersection-Flow-5K/labels/train/1662004766.172890902.txt  
  inflating: Intersection-Flow-5K/labels/train/1662004766.870704174.txt  
  inflating: Intersection-Flow-5K/labels/train/1662004768.070634604.txt  
  inflating: Intersection-Flow-5K/labels/train/1662004769.672873020.txt  
  inflating: Intersection-Flow-5K/labels/train/1662006309.754629135.txt  
  inflating: Intersection-Flow-5K/labels/train/1662006316.952394724.txt  
  inflating: Intersection-Flow-5K/labels/train/1662006318.055218697.txt  
  inflating: Intersection-Flow-5K/labels/train/1662006321.053

In [2]:
!pip install ultralytics

In [3]:
import torch
from torch.utils.tensorboard import SummaryWriter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from pathlib import Path
import random
import csv
import os
import cv2
import json
import glob
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

In [4]:
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [5]:
data_dir= "/content/Intersection-Flow-5K"

In [6]:
data = """
train: /content/Intersection-Flow-5K/images/train  # Path to training images
val: /content/Intersection-Flow-5K/images/val     # Path to validation images
test: /content/Intersection-Flow-5K/images/test   # Path to test images

# Class information
nc: 8
names: ['vehicle', 'bus', 'bicycle', 'pedestrian',
        'engine', 'truck', 'tricycle', 'obstacle']
"""

In [7]:
with open('/content/data.yaml', 'w') as file:
    file.write(data)

In [ ]:
def log_results_to_tensorboard(model_name, train_folder):
    results_path = Path(f'runs/detect/{train_folder}/results.csv')
    writer = SummaryWriter(log_dir=f"tensorboard/runs/{model_name}")

    if results_path.exists():
        import pandas as pd
        df = pd.read_csv(results_path)
        df.columns = df.columns.str.strip()

        for idx, row in df.iterrows():
            epoch = int(row['epoch']) if 'epoch' in row else idx
            if 'train/box_loss' in row:
                writer.add_scalar('Loss/train_box', row['train/box_loss'], epoch)
            if 'train/cls_loss' in row:
                writer.add_scalar('Loss/train_cls', row['train/cls_loss'], epoch)

            if 'metrics/precision(B)' in row:
                writer.add_scalar('Metrics/precision', row['metrics/precision(B)'], epoch)
            if 'metrics/recall(B)' in row:
                writer.add_scalar('Metrics/recall', row['metrics/recall(B)'], epoch)
            if 'metrics/mAP50(B)' in row:
                writer.add_scalar('Metrics/mAP50', row['metrics/mAP50(B)'], epoch)
            if 'metrics/mAP50-95(B)' in row:
                writer.add_scalar('Metrics/mAP50-95', row['metrics/mAP50-95(B)'], epoch)

        print("Метрики залогированы в TensorBoard")

        writer.close()

In [9]:
epochs = 5
batch_size = 16
lr = 0.001
momentum = 0.937
weight_decay=0.0005
device = 'cuda' if torch.cuda.is_available() else 'cpu'
save = True
plots = True
verbose = True
save_period = 10
data_yaml = '/content/data.yaml'

In [ ]:
model = YOLO('yolo11n')

In [ ]:
results = model.train(data=data_yaml, epochs=epochs, batch=batch_size, momentum=momentum,
                      device=device, save=save, verbose=verbose, lr0=lr, weight_decay=weight_decay, plots=plots, freeze=10)


Ultralytics 8.3.223 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretrained

In [ ]:
log_results_to_tensorboard('yolo11n','train')

Метрики залогированы в TensorBoard


In [ ]:
model_l = YOLO('yolo11l')
results_l = model_l.train(data=data_yaml, epochs=epochs, batch=batch_size, momentum=momentum,
                      device=device, save=save, verbose=verbose, lr0=lr, weight_decay=weight_decay, plots=plots, freeze=10)

Ultralytics 8.3.223 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11l.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretraine

In [ ]:
log_results_to_tensorboard('yolo11l','train2')

Метрики залогированы в TensorBoard


In [ ]:
model_n_val = YOLO('runs/detect/train/weights/best.pt')

metrics = model_n_val.val(data=data_yaml)

print(f"\nРезультаты валидации:")
print(f"   mAP50: {metrics.box.map50:.4f}")
print(f"   mAP50-95: {metrics.box.map:.4f}")
print(f"   Precision: {metrics.box.mp:.4f}")
print(f"   Recall: {metrics.box.mr:.4f}")

Ultralytics 8.3.223 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11n summary (fused): 100 layers, 2,583,712 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2176.5±1218.2 MB/s, size: 900.0 KB)
val: Scanning /content/Intersection-Flow-5K/labels/val.cache... 722 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 722/722 1.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 46/46 2.6it/s 17.6s
                   all        722      38868      0.624        0.3      0.348      0.232
               vehicle        722      23450      0.707      0.653      0.715      0.502
                   bus        226        315      0.755        0.4       0.47      0.378
               bicycle        585       2771      0.588      0.376      0.408        0.2
            pedestrian        531       2250      0.695      0.117      0.175     0.0724
                engine         66

In [ ]:
model_l_val = YOLO('runs/detect/train2/weights/best.pt')

metrics = model_l_val.val(data=data_yaml)

print(f"\nРезультаты валидации:")
print(f"   mAP50: {metrics.box.map50:.4f}")
print(f"   mAP50-95: {metrics.box.map:.4f}")
print(f"   Precision: {metrics.box.mp:.4f}")
print(f"   Recall: {metrics.box.mr:.4f}")

Ultralytics 8.3.223 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11l summary (fused): 190 layers, 25,285,480 parameters, 0 gradients, 86.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2991.4±279.6 MB/s, size: 760.1 KB)
val: Scanning /content/Intersection-Flow-5K/labels/val.cache... 722 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 722/722 1.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 46/46 1.8it/s 25.2s
                   all        722      38868      0.788      0.514      0.584      0.397
               vehicle        722      23450      0.882       0.76      0.831      0.637
                   bus        226        315      0.837      0.749      0.776      0.584
               bicycle        585       2771       0.77      0.606      0.682      0.378
            pedestrian        531       2250      0.749      0.276      0.355      0.168
                engine         6

yolo11l обучается медленнее yolo11n. Валидационные метрики yolo11l превосходят yolo11n

In [ ]:
model = YOLO('yolo11l')
results = model.train(data=data_yaml, epochs=epochs, batch=batch_size, momentum=momentum,
                      device=device, save=save, verbose=verbose, lr0=0.01, weight_decay=weight_decay, plots=plots, freeze=10)
model_val = YOLO('runs/detect/train3/weights/best.pt')

metrics = model_val.val(data=data_yaml)

print(f"\nРезультаты валидации:")
print(f"   mAP50: {metrics.box.map50:.4f}")
print(f"   mAP50-95: {metrics.box.map:.4f}")
print(f"   Precision: {metrics.box.mp:.4f}")
print(f"   Recall: {metrics.box.mr:.4f}")

Ultralytics 8.3.223 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11l.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretrained

In [ ]:
model = YOLO('yolo11l')
results = model.train(data=data_yaml, epochs=epochs, batch=batch_size, momentum=momentum,
                      device=device, save=save, imgsz=800, verbose=verbose, lr0=lr, weight_decay=weight_decay, plots=plots, freeze=10)
model_val = YOLO('runs/detect/train4/weights/best.pt')

metrics = model_val.val(data=data_yaml)

print(f"\nРезультаты валидации:")
print(f"   mAP50: {metrics.box.map50:.4f}")
print(f"   mAP50-95: {metrics.box.map:.4f}")
print(f"   Precision: {metrics.box.mp:.4f}")
print(f"   Recall: {metrics.box.mr:.4f}")

Ultralytics 8.3.223 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11l.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretraine

In [ ]:
model = YOLO('yolo11l')
results = model.train(data=data_yaml, epochs=epochs, batch=batch_size, momentum=momentum,
                      device=device, save=save, verbose=verbose, lr0=lr, weight_decay=weight_decay, plots=plots, freeze=5)
model_val = YOLO('runs/detect/train5/weights/best.pt')

metrics = model_val.val(data=data_yaml)

print(f"\nРезультаты валидации:")
print(f"   mAP50: {metrics.box.map50:.4f}")
print(f"   mAP50-95: {metrics.box.map:.4f}")
print(f"   Precision: {metrics.box.mp:.4f}")
print(f"   Recall: {metrics.box.mr:.4f}")

Ultralytics 8.3.223 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=5, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11l.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train5, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretrained

Лучше всего показала себя модель с увеличенным imgsz

In [ ]:
best_model = YOLO('runs/detect/train4/weights/best.pt')
best_model_result = best_model.predict('/content/Intersection-Flow-5K/images/val',
                                       name='preds', save_txt=True, save_conf=True, conf=0.001, half=True, batch=16)


image 1/722 /content/Intersection-Flow-5K/images/val/10110038.jpg: 480x800 61 vehicles, 37 bicycles, 5 pedestrians, 1 obstacle, 14.1ms
image 2/722 /content/Intersection-Flow-5K/images/val/10110119.jpg: 480x800 68 vehicles, 42 bicycles, 2 pedestrians, 1 truck, 1 obstacle, 14.1ms
image 3/722 /content/Intersection-Flow-5K/images/val/10110146.jpg: 480x800 66 vehicles, 4 buss, 72 bicycles, 3 pedestrians, 1 truck, 1 tricycle, 1 obstacle, 14.1ms
image 4/722 /content/Intersection-Flow-5K/images/val/1011016.jpg: 480x800 103 vehicles, 4 buss, 2 bicycles, 3 trucks, 1 obstacle, 14.1ms
image 5/722 /content/Intersection-Flow-5K/images/val/10110213.jpg: 480x800 63 vehicles, 59 bicycles, 9 pedestrians, 5 trucks, 1 obstacle, 14.1ms
image 6/722 /content/Intersection-Flow-5K/images/val/10110281.jpg: 480x800 63 vehicles, 1 bus, 59 bicycles, 7 pedestrians, 1 truck, 1 obstacle, 14.1ms
image 7/722 /content/Intersection-Flow-5K/images/val/10110362.jpg: 480x800 66 vehicles, 34 bicycles, 7 pedestrians, 1 engin

In [ ]:
def yolo_to_coco_gt(images_dir, labels_dir, output_json='gt_coco.json'):
    images = []
    annotations = []
    categories = [
        {"id": 0, "name": "vehicle"},
        {"id": 1, "name": "bus"},
        {"id": 2, "name": "bicycle"},
        {"id": 3, "name": "pedestrian"},
        {"id": 4, "name": "engine"},
        {"id": 5, "name": "truck"},
        {"id": 6, "name": "tricycle"},
        {"id": 7, "name": "obstacle"},
    ]

    ann_id = 1

    image_files = sorted(glob.glob(os.path.join(images_dir, "*.jpg")))
    for img_id, img_path in enumerate(image_files):
        file_name = os.path.basename(img_path)
        label_path = os.path.join(labels_dir, file_name.replace(".jpg", ".txt"))
        if not os.path.exists(label_path):
            continue

        with Image.open(img_path) as im:
            w, h = im.size

        images.append({
            "id": img_id,
            "file_name": file_name,
            "width": w,
            "height": h
        })

        with open(label_path, "r") as f:
            lines = f.readlines()

        for line in lines:
            cls, x, y, bw, bh = map(float, line.strip().split())
            cls = int(cls)
            x1 = (x - bw / 2) * w
            y1 = (y - bh / 2) * h
            bw *= w
            bh *= h
            annotations.append({
                "id": ann_id,
                "image_id": img_id,
                "category_id": cls,
                "bbox": [x1, y1, bw, bh],
                "area": bw * bh,
                "iscrowd": 0
            })
            ann_id += 1

    coco_dict = {
        "info": "",
        "images": images,
        "annotations": annotations,
        "categories": categories
    }

    with open(output_json, "w") as f:
        json.dump(coco_dict, f)
    print(f"COCO GT saved: {output_json} ({len(images)} images, {len(annotations)} boxes)")

In [ ]:
def yolo_pred_to_coco_pred(images_dir, labels_dir, output_json='preds_coco.json'):
    preds = []
    image_files = sorted(glob.glob(os.path.join(images_dir, "*.jpg")))

    id_map = {os.path.basename(p): i for i, p in enumerate(image_files)}

    for img_path in image_files:
        file_name = os.path.basename(img_path)
        img_id = id_map[file_name]

        label_path = os.path.join(labels_dir, file_name.replace(".jpg", ".txt"))
        if not os.path.exists(label_path):
            continue

        with Image.open(img_path) as im:
            w, h = im.size

        with open(label_path, "r") as f:
            lines = f.readlines()

        for line in lines:
            parts = list(map(float, line.strip().split()))
            if len(parts) < 6:
                continue
            cls, x, y, bw, bh, conf = parts
            x1 = (x - bw / 2) * w
            y1 = (y - bh / 2) * h
            bw *= w
            bh *= h
            preds.append({
                "image_id": img_id,
                "category_id": int(cls),
                "bbox": [x1, y1, bw, bh],
                "score": conf
            })

    with open(output_json, "w") as f:
        json.dump(preds, f)
    print(f"COCO preds saved: {output_json} ({len(preds)} boxes)")

In [ ]:
yolo_to_coco_gt("/content/Intersection-Flow-5K/images/val",'/content/Intersection-Flow-5K/labels/val')

COCO GT saved: gt_coco.json (722 images, 38868 boxes)


In [ ]:
yolo_pred_to_coco_pred("/content/Intersection-Flow-5K/images/val",'/content/runs/detect/preds/labels')

COCO preds saved: preds_coco.json (113528 boxes)


In [ ]:
def evaluate_with_pycocotools(gt_path, pred_path):
    coco_gt = COCO(gt_path)

    coco_dt = coco_gt.loadRes(pred_path)

    coco_eval = COCOeval(coco_gt, coco_dt, iouType='bbox')

    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    metrics = {
        'mAP': coco_eval.stats[0],
        'mAP50': coco_eval.stats[1],
        'mAP75': coco_eval.stats[2],
        'mAP_small': coco_eval.stats[3],
        'mAP_medium': coco_eval.stats[4],
        'mAP_large': coco_eval.stats[5],
    }

    print("\n Детальные метрики:")
    for metric_name, value in metrics.items():
        print(f" {metric_name}: {value:.4f}")

    return metrics

In [ ]:
metrics_coco = evaluate_with_pycocotools('/content/gt_coco.json', '/content/preds_coco.json')

loading annotations into memory...
Done (t=0.12s)
creating index...
index created!
Loading and preparing results...
DONE (t=1.06s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=40.95s).
Accumulating evaluation results...
DONE (t=1.65s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.440
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.623
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.465
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.129
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.455
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.723
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.265
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.428
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDet

In [ ]:
model_val = YOLO('runs/detect/train4/weights/best.pt')

metrics = model_val.val(data=data_yaml)

print(f"\nРезультаты валидации:")
print(f"   mAP50: {metrics.box.map50:.4f}")
print(f"   mAP50-95: {metrics.box.map:.4f}")

Ultralytics 8.3.223 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11l summary (fused): 190 layers, 25,285,480 parameters, 0 gradients, 86.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1878.8±254.4 MB/s, size: 760.1 KB)
val: Scanning /content/Intersection-Flow-5K/labels/val.cache... 722 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 722/722 468.1Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 46/46 1.5it/s 31.1s
                   all        722      38868      0.785      0.597      0.673      0.472
               vehicle        722      23450      0.873      0.826      0.877      0.693
                   bus        226        315       0.78      0.784      0.801      0.651
               bicycle        585       2771      0.745      0.717      0.771      0.457
            pedestrian        531       2250      0.798      0.328      0.477      0.238
                engine        

Метрики полученные через pycocotool немного отличаются от ultralytics

In [ ]:
model = YOLO('yolo11l')
results = model.train(data=data_yaml, epochs=10, batch=8, momentum=momentum,
                      device=device, save=save, imgsz=1024, verbose=verbose, lr0=lr, weight_decay=weight_decay, plots=plots, freeze=10)
model_val = YOLO('runs/detect/train6/weights/best.pt')

metrics = model_val.val(data=data_yaml)

print(f"\nРезультаты валидации:")
print(f"   mAP50: {metrics.box.map50:.4f}")
print(f"   mAP50-95: {metrics.box.map:.4f}")
print(f"   Precision: {metrics.box.mp:.4f}")
print(f"   Recall: {metrics.box.mr:.4f}")

Ultralytics 8.3.223 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11l.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train7, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretrain

In [ ]:
best_model = YOLO('runs/detect/train7/weights/best.pt')
best_model_result = best_model.predict('/content/Intersection-Flow-5K/images/val',
                                       name='preds', save_txt=True, save_conf=True, conf=0.001, half=True, batch=16)
yolo_pred_to_coco_pred("/content/Intersection-Flow-5K/images/val",'/content/runs/detect/preds/labels')

In [ ]:
%load_ext tensorboard
%tensorboard --logdir=/content/tensorboard/runs

<IPython.core.display.Javascript object>

In [ ]:
!zip -r yolo.zip runs tensorboard gt_coco.json preds_coco.json

  adding: runs/ (stored 0%)
  adding: runs/detect/ (stored 0%)
  adding: runs/detect/train3/ (stored 0%)
  adding: runs/detect/train3/labels.jpg (deflated 33%)
  adding: runs/detect/train3/BoxP_curve.png (deflated 6%)
  adding: runs/detect/train3/results.png (deflated 8%)
  adding: runs/detect/train3/weights/ (stored 0%)
  adding: runs/detect/train3/weights/best.pt (deflated 8%)
  adding: runs/detect/train3/weights/last.pt (deflated 8%)
  adding: runs/detect/train3/val_batch2_labels.jpg (deflated 3%)
  adding: runs/detect/train3/confusion_matrix_normalized.png (deflated 17%)
  adding: runs/detect/train3/args.yaml (deflated 52%)
  adding: runs/detect/train3/train_batch1.jpg (deflated 3%)
  adding: runs/detect/train3/BoxF1_curve.png (deflated 6%)
  adding: runs/detect/train3/results.csv (deflated 57%)
  adding: runs/detect/train3/val_batch1_labels.jpg (deflated 3%)
  adding: runs/detect/train3/val_batch0_pred.jpg (deflated 4%)
  adding: runs/detect/train3/confusion_matrix.png (deflated 1

In [ ]:
import gc

In [ ]:
gc.collect()

54719